# Byte I/O - Python

All 24 Python examples from [docs/io.md](https://platob.github.io/yggdryl/io/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import IOBase

handle = IOBase.from_bytes()
handle.pwrite(0, b"symbol,price\n")
handle.pwrite(13, b"AAPL,1\n")
assert handle.size == 20

# Two reads at different offsets, in any order: there is no shared cursor.
assert handle.read_range_bytes(13, 4) == b"AAPL"
assert handle.read_range_bytes(0, 6) == b"symbol"

## Streamed bytes

In [ ]:
from yggdryl import IOBase

handle = IOBase.from_bytes(b"0123456789")
assert list(handle.pstream_bytes(2, 3)) == [b"234", b"567", b"89"]

cursor = handle.cursor(1)
stream = cursor.stream_bytes(2)
assert next(stream) == b"12"
assert cursor.tell() == 3

## Built from what you already hold

In [ ]:
import io
import pathlib
import tempfile

from yggdryl import IOBase

# An open file names its own location, so the handle addresses the path.
target = pathlib.Path(tempfile.mkdtemp()) / "quotes.json"
target.write_bytes(b"{}")
with open(target, "rb") as stream:
    handle = IOBase(stream)
assert handle.name == "quotes.json"

# A nameless stream holds only content, so the content is what is taken.
buffered = IOBase(io.BytesIO(b'{"symbol": "AAPL"}'))
buffered.media_type = "application/json"
assert buffered.read_text() == '{"symbol": "AAPL"}'

## Laziness

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())

# Constructing touches nothing: no file is created, opened, or mapped.
handle = IOBase(root / "nested" / "lazy.csv")
assert not handle.exists()

# Reading something absent yields nothing rather than raising.
assert handle.size == 0
assert handle.read_bytes() == b""

# Writing creates the resource, and any parent it needs.
handle.write_text("symbol,price\n")
assert handle.is_file()
assert handle.read_text() == "symbol,price\n"

## Kinds

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

folder = IOBase(pathlib.Path(tempfile.mkdtemp()))
assert folder.is_dir()
assert not folder.is_file()

# Nothing is there, so nothing has decided; a write settles it.
leaf = folder / "ticks.csv"
assert not leaf.exists()
leaf.write_text("symbol\n")
assert leaf.is_file()
assert not leaf.is_dir()

## Bytes or rows

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

root = IOBase(pathlib.Path(tempfile.mkdtemp()))

notes = root / "notes.txt"
assert notes.is_atomic()
assert not notes.is_tabular()

# The name is enough: nothing has been written to this location yet.
trades = root / "trades.parquet"
assert trades.is_tabular()
assert not trades.is_atomic()

assert not root.is_atomic()

## Whole values

In [ ]:
from yggdryl import IOBase

handle = IOBase.from_bytes()
handle.write_bytes(b"symbol,price\n")

# `append_bytes` reports the offset the bytes landed at.
assert handle.append_bytes(b"AAPL,1\n") == 13
assert handle.read_range_bytes(0, 6) == b"symbol"
# A range past the end yields what exists rather than raising.
assert handle.read_range_bytes(100, 4) == b""
assert len(handle.read_bytes()) == 20

## Structured values

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase, Scalar

path = pathlib.Path(tempfile.mkdtemp()) / "trade.json.gz"
handle = IOBase(path)
handle.write_scalar({"quantity": 2, "symbol": "AAPL"})
field = "trade: struct<quantity: int32 not null, symbol: utf8 not null> not null"
assert handle.read_scalar(field) == {"quantity": 2, "symbol": "AAPL"}
value = handle.read_scalar(field, cls=Scalar)
assert value.kind == "sequence"

## Cursors

In [ ]:
from yggdryl import IOBase

handle = IOBase.from_bytes()
cursor = handle.cursor()
cursor.write(b"symbol,price\n")

# The write landed on the handle itself; the position is the cursor's.
assert handle.read_bytes() == b"symbol,price\n"
assert cursor.seek(-6, 2) == 7
assert cursor.read(5) == b"price"

## What the bytes are

In [ ]:
from yggdryl import IOBase

# Nothing names an in-memory buffer, so its type comes from its bytes.
handle = IOBase.from_bytes(b'{"symbol":"AAPL"}')
assert str(handle.media_type.base) == "application/json"

# It is re-derived after the content changes.
handle.write_bytes(b"PAR1payload")
assert str(handle.media_type.base) == "application/vnd.apache.parquet"

## Adding and removing a coding

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())
plain = IOBase(root / "rows.json")
plain.write_bytes(b'{"symbol":"AAPL"}')

# Nothing wraps these bytes, so there is nothing to undo.
assert plain.codec is None

encoded = IOBase(root / "rows.json.gz")
assert encoded.codec == "gzip"

# The target's name already said gzip, so nothing here repeats it.
assert plain.compress_into(encoded) == encoded.size
assert encoded.read_bytes()[:2] == b"\x1f\x8b"

decoded = IOBase(root / "roundtrip.json")
assert encoded.decompress_into(decoded) == 17
assert decoded.read_bytes() == plain.read_bytes()
assert decoded.codec is None

# A target declaring no coding is refused rather than copied unchanged.
reason = None
try:
    plain.compress_into(IOBase(root / "copy.json"))
except ValueError as error:
    reason = str(error)
assert "expected a target declaring a content coding" in reason
assert not (root / "copy.json").exists()

## Open and close

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

path = pathlib.Path(tempfile.mkdtemp()) / "trades.csv"

# `with` is the scoped pair: `__enter__` opens and `__exit__` closes.
with IOBase(path) as handle:
    handle.write_text("symbol,price\n")
    assert handle.opened
    assert not handle.closed

# Closing published the bytes at their exact length, which is what another
# reader needs; the handle stays usable and simply re-materializes.
assert path.stat().st_size == 13
assert IOBase(path).read_text() == "symbol,price\n"

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

target = pathlib.Path(tempfile.mkdtemp()) / "lake" / "trades.parquet"
IOBase(target).overwrite_arrow_table(pa.table({"id": [1, 2], "venue": ["XNAS", "XNYS"]}))

# Metadata-heavy work belongs inside the scope: the schema probe, the
# per-batch reads, and the size checks all reuse what `open` cached, and
# `close` releases it at a known point.
rows = 0
with IOBase(target) as handle:
    field = handle.read_arrow_field()
    for batch in handle.read_arrow_reader():
        rows += batch.num_rows
assert rows == 2

# Outside a scope the same calls still work - each one just fetches fresh,
# which is exactly right for a resource another writer may be changing.
assert IOBase(target).read_arrow_field() == field

## Clearing and removing

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())
handle = IOBase(root / "logs")
handle.mkdir()
(handle / "a.log").write_text("line\n")

# Clearing empties the container and keeps it.
handle.clear()
assert list(handle.iterdir()) == []
assert handle.is_dir()

# Removing deletes it; a second call succeeds, having done nothing. A
# handle asked for as a container keeps answering `is_dir`, because that
# is what it was asked for - the parent's listing is what shows it gone.
handle.remove()
handle.remove()
assert list(IOBase(root).iterdir()) == []

# A container that still has children is refused rather than recursed into.
handle.mkdir()
(handle / "a.log").write_text("line\n")
try:
    handle.remove()
except Exception as error:
    assert "children" in str(error)
handle.remove(recursive=True)
assert list(IOBase(root).iterdir()) == []

# The handle stays usable and lazy - a write recreates the resource.
leaf = IOBase(root / "trades.csv")
leaf.write_text("symbol,price\n")
leaf.remove()
assert not leaf.exists()
leaf.write_text("symbol,price\n")
assert leaf.read_text() == "symbol,price\n"

## Arrow batches

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

# A PyArrow schema is the schema; the binding imports it once at the boundary.
schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
batch = pa.record_batch({"id": [1, 2], "symbol": ["AAPL", None]}, schema=schema)

# The handle's own media type picks the encoding; no format argument is passed.
handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
options = handle.record_options()

# The write path takes a batch reader and nothing else.
handle.overwrite_arrow_batch(batch, options=options)
assert handle.read_arrow_field(options=options).name == "row"

# The read path returns one. Batches arrive one at a time, never as a vector.
rows = sum(part.num_rows for part in handle.read_arrow_reader(options=options))
assert rows == 2

### Canonical record-write signatures

In [ ]:
import pathlib
import tempfile

import pytest

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())

# An absent resource holds no batches rather than failing to parse.
empty = IOBase(root / "absent.arrows")
assert empty.read_arrow_reader().read_all().num_rows == 0

# An encoding this build does not implement is named rather than guessed.
csv = IOBase(root / "trades.csv")
with pytest.raises(ValueError, match="text/csv"):
    csv.record_options()

## Lazy scans

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

target = pathlib.Path(tempfile.mkdtemp()) / "trades.parquet"
IOBase(target).overwrite_arrow_table(pa.table({"symbol": ["AAPL", "MSFT"], "price": [187.23, 402.11]}))

# A local Parquet leaf becomes the real lazy scan - projection and
# predicate pushdown belong to the engine, and the handle publishes its
# bytes at their exact length first so the foreign reader sees a whole file.
lazy = IOBase(target).scan_polars()
assert lazy.select("symbol").head(10).collect().height == 2

# The pyarrow spelling of the same idea, as a dataset Scanner.
scanner = IOBase(target).scan_arrow()
assert scanner.to_table().num_rows == 2

# Anything a foreign scanner cannot mmap - an in-memory buffer, a
# compressed name, an Arrow stream - streams through the native reader
# instead, so both calls answer for every holder.
memory = IOBase.from_bytes()
memory.media_type = "application/vnd.apache.arrow.stream"
memory.overwrite_arrow_table(pa.table({"symbol": ["AAPL"]}))
assert memory.scan_arrow().to_table().num_rows == 1

## Column pushdown

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

stored = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string(), nullable=False),
    pa.field("venue", pa.string(), nullable=False),
])
batch = pa.record_batch(
    {"id": [1, 2], "symbol": ["AAPL", "MSFT"], "venue": ["XNAS", "XNAS"]},
    schema=stored,
)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
handle.overwrite_arrow_batch(batch)

# One of the three columns, declared as this read's schema.
options = handle.record_options()
options.field = pa.schema([pa.field("id", pa.int64(), nullable=False)])

projected = handle.read_arrow_reader(options=options)
assert projected.schema.names == ["id"]
assert projected.read_all().num_columns == 1

# The resource is unchanged: it still holds all three.
assert len(handle.read_arrow_field().dtype) == 3

# A column it does not hold cannot be projected out of it, so the encoding
# reads everything and the cast supplies that column as nulls.
options.field = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("nowhere", pa.string()),
])
widened = handle.read_arrow_reader(options=options)
assert widened.schema.names == ["id", "nowhere"]

## Limiting a read or a write

In [ ]:
import pyarrow as pa

from yggdryl import IOBase

handle = IOBase.from_bytes()
handle.media_type = "application/vnd.apache.arrow.stream"
handle.overwrite_arrow_table(pa.table({"id": list(range(1_000))}))

# Ten result rows, exactly: the batch the bound lands inside is sliced.
ten = handle.record_options()
ten.max_row_size = 10
assert handle.read_arrow_reader(options=ten).read_all().num_rows == 10

# Zero is a valid ask: the shaped schema answers, and no batch flows.
zero = handle.record_options()
zero.max_row_size = 0
empty = handle.read_arrow_reader(options=zero)
assert empty.schema.names == ["id"]
assert empty.read_all().num_rows == 0

# A non-zero byte bound always yields at least one row.
one_byte = handle.record_options()
one_byte.max_byte_size = 1
assert handle.read_arrow_reader(options=one_byte).read_all().num_rows == 1

# A limited write truncates the data the caller offered: three rows land,
# and what the bound cut off is never pulled from the reader.
copy = IOBase.from_bytes()
copy.media_type = "application/vnd.apache.arrow.stream"
first_three = copy.record_options()
first_three.max_row_size = 3
copy.overwrite_arrow_reader(handle.read_arrow_reader(), options=first_three)
assert copy.read_arrow_reader().read_all().num_rows == 3

## Appending and merging

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
rows = lambda ids, symbols: pa.record_batch(
    {"id": ids, "symbol": symbols}, schema=schema
)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
options = handle.record_options()
options.field = schema

# No match key: the resource is replaced.
handle.overwrite_arrow_batch(rows([1, 2], ["AAPL", "MSFT"]), options=options)

# Appending reads what is there, chains the new batches after it, and rewrites.
handle.append_arrow_batch(rows([3], ["NVDA"]), options=options)
assert handle.read_arrow_reader(options=options).read_all().num_rows == 3

# A match key merges: `2` is already stored and updates, `9` is new and appends.
merging = handle.record_options()
merging.field = schema
merging.merge_by_names = ["id"]
handle.merge_arrow_batch(rows([2, 9], ["MSFT.O", "AMD"]), options=merging)
assert handle.read_arrow_reader(options=options).read_all().num_rows == 4

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "orders.arrows")
handle.overwrite_arrow_table(pa.table({"id": [1, 2], "symbol": ["AAPL", "MSFT"]}))

# Record settings live on the one options object shared by every operation.
options = handle.record_options()
options.select_by_names = ["symbol"]
narrowed = handle.read_arrow_reader(options=options).read_all()
assert narrowed.column_names == ["symbol"]

## Globbing and Hive partitions

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp()) / "lake"
for year in ("2024", "2025"):
    leaf = root / f"year={year}" / "month=01"
    leaf.mkdir(parents=True)
    (leaf / "part-0.parquet").write_bytes(b"parquet")

lake = IOBase(root)

assert len(list(lake.glob("year=2024/**/*.parquet"))) == 1
assert len(list(lake.rglob("*.parquet"))) == 2

selected = list(lake.children_where({"year": "2024"}))
assert len(selected) == 1
assert selected[0].partitions == (("year", "2024"), ("month", "01"))

## Partition pruning and filtering

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp()) / "lake"
IOBase(root / "year=2024" / "month=01" / "trades.arrows").overwrite_arrow_table(
    pa.table({"id": [1, 2]})
)
IOBase(root / "year=2024" / "month=02" / "trades.arrows").overwrite_arrow_table(
    pa.table({"id": [3]})
)

lake = IOBase(root)
options = lake.record_options()
options.filter_partitions = [("year", "2024"), ("month", "01")]
reader = lake.read_arrow_reader(options=options)
assert reader.read_all().num_rows == 2

## Partition columns in the data

In [ ]:
import pathlib
import shutil
import tempfile

import pyarrow as pa

from yggdryl import IOBase, RecordOptions

root = pathlib.Path(tempfile.mkdtemp())
(root / "year=2024" / "month=01").mkdir(parents=True)

schema = pa.schema([
    pa.field("price", pa.int64(), nullable=False),
    pa.field("year", pa.int32(), nullable=False),
    pa.field("month", pa.string(), nullable=False),
])
batch = pa.record_batch(
    {"price": [10, 20], "year": [2024, 2024], "month": ["01", "01"]},
    schema=schema,
)

# The rows carry every column; the write drops the two the path spells out.
lake = IOBase(root)
options = RecordOptions("part.arrows")
options.field = schema
lake.overwrite_arrow_batch(batch, options=options)

# Only `price` reached the leaf; the other two are the directory names.
leaf = lake / "year=2024" / "month=01" / "part-0.arrows"
assert len(leaf.read_arrow_field().dtype) == 1

# Reading the folder restores them with their declared types.
restored = lake.read_arrow_reader(options=options).read_all()
assert restored.column_names == ["price", "year", "month"]
assert restored.schema.field("year").type == pa.int32()

shutil.rmtree(root)